# Remote Operations Pressure Analysis

## Objective
Analyze well performance by comparing actual vs planned pressure and identify anomalies or underperforming wells.

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

well = pd.read_csv('well_info.csv')
treatment = pd.read_csv('treatment_data.csv')
equipment = pd.read_csv('equipment_data.csv')
materials = pd.read_csv('materials.csv')
plan = pd.read_csv('job_plan.csv')
dictionary = pd.read_csv('data_dictionary.csv')

print("Data loaded successfully")
treatment.head()

Data loaded successfully


,timestamp,well_id,stage,treating_pressure_psi,slurry_rate_bpm,proppant_conc_lbgal,planned_pressure_psi,planned_slurry_rate_bpm,pressure_deviation_psi,is_pumping,known_injected_issue
0,2026-01-15 06:00:00,W-101,1,8196.1,86.92,3.05,8423.7,87.9,-227.6,1,none
1,2026-01-15 06:01:00,W-101,1,8830.6,83.19,3.00,8423.7,87.9,406.9,1,none
2,2026-01-15 06:02:00,W-101,1,8405.7,87.20,3.16,8423.7,87.9,-18.0,1,none
3,2026-01-15 06:03:00,W-101,1,8247.2,91.01,2.99,8423.7,87.9,-176.5,1,none
4,2026-01-15 06:04:00,W-101,1,8255.5,87.26,2.64,8423.7,87.9,-168.2,1,none


In [33]:
# Data integrity checks

print("Missing values:\n", treatment.isnull().sum())
print("\nDuplicate rows:", treatment.duplicated().sum())

Missing values:
 timestamp                  0
well_id                    0
stage                      0
treating_pressure_psi      0
slurry_rate_bpm            0
proppant_conc_lbgal        0
planned_pressure_psi       0
planned_slurry_rate_bpm    0
pressure_deviation_psi     0
is_pumping                 0
known_injected_issue       0
dtype: int64

Duplicate rows: 0


In [34]:
# Check for unrealistic values

bad_pressure = treatment[treatment['treating_pressure_psi'] < 0]
high_pressure = treatment[treatment['treating_pressure_psi'] > 15000]

print("Bad pressure rows:", len(bad_pressure))
print("High pressure rows:", len(high_pressure))

Bad pressure rows: 0
High pressure rows: 0


In [35]:
# Detect pressure spikes

treatment['pressure_diff'] = treatment['treating_pressure_psi'].diff()

spikes = treatment[treatment['pressure_diff'].abs() > 500]

print("Pressure spike events:", len(spikes))
spikes.head()

Pressure spike events: 178


,timestamp,well_id,stage,treating_pressure_psi,slurry_rate_bpm,proppant_conc_lbgal,planned_pressure_psi,planned_slurry_rate_bpm,pressure_deviation_psi,is_pumping,known_injected_issue,pressure_diff
1,2026-01-15 06:01:00,W-101,1,8830.6,83.19,3.00,8423.7,87.9,406.9,1,none,634.5
11,2026-01-15 06:11:00,W-101,1,10553.1,85.15,2.99,8423.7,87.9,2129.5,1,pressure_spike,1851.6
12,2026-01-15 06:12:00,W-101,1,9797.4,86.31,2.99,8423.7,87.9,1373.7,1,none,-755.7
13,2026-01-15 06:13:00,W-101,1,10375.4,83.74,3.03,8423.7,87.9,1951.8,1,pressure_spike,578.0
16,2026-01-15 06:16:00,W-101,1,8143.2,90.86,3.10,8423.7,87.9,-280.5,1,none,-2086.5


In [36]:
# Smarter anomaly detection using statistics

mean_diff = treatment['pressure_diff'].mean()
std_diff = treatment['pressure_diff'].std()

threshold = mean_diff + 3 * std_diff

smart_spikes = treatment[treatment['pressure_diff'].abs() > threshold]

print("Dynamic threshold:", threshold)
print("Smart spike events:", len(smart_spikes))
smart_spikes.head()

Dynamic threshold: 1969.3637908204792
Smart spike events: 17


,timestamp,well_id,stage,treating_pressure_psi,slurry_rate_bpm,proppant_conc_lbgal,planned_pressure_psi,planned_slurry_rate_bpm,pressure_deviation_psi,is_pumping,known_injected_issue,pressure_diff
16,2026-01-15 06:16:00,W-101,1,8143.2,90.86,3.10,8423.7,87.9,-280.5,1,none,-2086.5
233,2026-01-15 12:00:00,W-101,4,7834.4,81.09,3.49,7849.8,81.7,-15.4,1,none,-2874.2
313,2026-01-15 14:00:00,W-101,5,10859.5,86.84,3.45,10801.5,88.8,58.1,1,none,3106.1
463,2026-01-17 06:33:00,W-102,1,13700.7,93.04,1.17,11013.1,84.3,2687.6,1,pressure_spike,2879.8
468,2026-01-17 06:38:00,W-102,1,10891.7,84.26,1.28,11013.1,84.3,-121.4,1,none,-2439.5


In [37]:
# Compare planned vs actual pressure

treatment['pressure_error'] = (
    treatment['treating_pressure_psi'] - treatment['planned_pressure_psi']
)

print("Average pressure error:", treatment['pressure_error'].mean())
print("Max pressure error:", treatment['pressure_error'].max())
print("Min pressure error:", treatment['pressure_error'].min())

Average pressure error: -92.9092893636786
Max pressure error: 2922.199999999999
Min pressure error: -9988.5


In [38]:
# Find worst performing rows

worst_cases = treatment.sort_values('pressure_error').head(10)

worst_cases[['well_id', 'stage', 'treating_pressure_psi', 'planned_pressure_psi', 'pressure_error']]

,well_id,stage,treating_pressure_psi,planned_pressure_psi,pressure_error
1690,W-104,6,0.0,9988.5,-9988.5
1691,W-104,6,0.0,9988.5,-9988.5
1688,W-104,6,0.0,9988.5,-9988.5
1689,W-104,6,0.0,9988.5,-9988.5
1687,W-104,6,0.0,9988.5,-9988.5
1692,W-104,6,0.0,9988.5,-9988.5
606,W-102,3,0.0,9590.7,-9590.7
607,W-102,3,0.0,9590.7,-9590.7
611,W-102,3,0.0,9590.7,-9590.7
610,W-102,3,0.0,9590.7,-9590.7


In [39]:
# Group by well to find problem wells

well_performance = treatment.groupby('well_id')['pressure_error'].mean()

well_performance.sort_values()

well_id
W-104   -241.126339
W-102   -233.941119
W-103     -4.685442
W-105     -0.158427
W-101     14.359070
Name: pressure_error, dtype: float64

In [40]:
treatment[treatment['treating_pressure_psi'] == 0][
    ['well_id', 'stage', 'timestamp']
].value_counts()

well_id  stage  timestamp          
W-102    3      2026-01-17 10:13:00    1
                2026-01-17 10:14:00    1
                2026-01-17 10:15:00    1
                2026-01-17 10:16:00    1
                2026-01-17 10:17:00    1
                2026-01-17 10:18:00    1
         5      2026-01-17 14:23:00    1
                2026-01-17 14:24:00    1
                2026-01-17 14:25:00    1
                2026-01-17 14:26:00    1
                2026-01-17 14:27:00    1
                2026-01-17 14:28:00    1
W-104    2      2026-01-21 08:12:00    1
                2026-01-21 08:13:00    1
                2026-01-21 08:14:00    1
                2026-01-21 08:15:00    1
                2026-01-21 08:16:00    1
                2026-01-21 08:17:00    1
         6      2026-01-21 16:42:00    1
                2026-01-21 16:43:00    1
                2026-01-21 16:44:00    1
                2026-01-21 16:45:00    1
                2026-01-21 16:46:00    1
                2026-

In [41]:
# Check if zero pressure is clustered in time
zero_pressure = treatment[treatment['treating_pressure_psi'] == 0]

zero_pressure.groupby(['well_id', 'stage']).size()

well_id  stage
W-102    3        6
         5        6
W-104    2        6
         6        6
dtype: int64

In [42]:
treatment_clean = treatment[treatment['treating_pressure_psi'] > 0].copy()

# Diagnostic checks
print(f"Rows in treatment_clean: {len(treatment_clean)}")
print(f"Columns: {treatment_clean.columns.tolist()}")
print(f"well_id unique values: {treatment_clean['well_id'].nunique() if len(treatment_clean) > 0 else 'N/A'}")

Rows in treatment_clean: 2129
Columns: ['timestamp', 'well_id', 'stage', 'treating_pressure_psi', 'slurry_rate_bpm', 'proppant_conc_lbgal', 'planned_pressure_psi', 'planned_slurry_rate_bpm', 'pressure_deviation_psi', 'is_pumping', 'known_injected_issue', 'pressure_diff', 'pressure_error']
well_id unique values: 5


In [43]:
treatment_clean['pressure_error'] = (
    treatment_clean['treating_pressure_psi'] - 
    treatment_clean['planned_pressure_psi']
)

In [44]:
treatment_clean.sort_values('pressure_error').head(10)

,timestamp,well_id,stage,treating_pressure_psi,slurry_rate_bpm,proppant_conc_lbgal,planned_pressure_psi,planned_slurry_rate_bpm,pressure_deviation_psi,is_pumping,known_injected_issue,pressure_diff,pressure_error
1551,2026-01-21 12:55:00,W-104,4,7582.5,75.09,1.62,8238.7,74.6,-656.1,1,none,-376.9,-656.2
189,2026-01-15 10:20:00,W-101,3,10049.7,85.33,1.38,10655.4,82.9,-605.7,1,none,-589.8,-605.7
287,2026-01-15 12:54:00,W-101,4,7291.9,82.21,3.38,7849.8,81.7,-557.9,1,none,-441.7,-557.9
1056,2026-01-19 12:13:00,W-103,4,9653.4,75.20,2.79,10199.8,74.7,-546.4,1,none,-419.6,-546.4
704,2026-01-17 12:37:00,W-102,4,9340.4,80.79,2.03,9869.1,84.0,-528.8,1,none,-769.8,-528.7
1623,2026-01-21 14:41:00,W-104,5,7920.7,84.03,2.57,8448.8,82.1,-528.1,1,none,-451.3,-528.1
273,2026-01-15 12:40:00,W-101,4,7322.7,78.89,3.52,7849.8,81.7,-527.1,1,none,-607.4,-527.1
1886,2026-01-23 10:19:00,W-105,3,8771.4,77.32,2.04,9278.3,74.9,-506.9,1,none,-562.9,-506.9
1330,2026-01-21 07:10:00,W-104,1,7493.9,85.10,2.72,8000.6,86.5,-506.8,1,none,-513.3,-506.7
105,2026-01-15 08:12:00,W-101,2,8223.4,73.00,2.10,8723.3,77.3,-500.0,1,none,-255.2,-499.9


In [45]:
# Check for valid data before grouping
if len(treatment_clean) > 0 and 'well_id' in treatment_clean.columns:
    result = treatment_clean.groupby('well_id')['pressure_error'].mean().sort_values()
    print(result)
else:
    print("Warning: treatment_clean is empty or missing 'well_id' column")

well_id
W-103    -4.685442
W-105    -0.158427
W-104    11.943578
W-101    14.359070
W-102    32.559900
Name: pressure_error, dtype: float64


In [46]:
# Summary of average pressure error by well
well_stats = treatment_clean.groupby('well_id')['pressure_error'].agg(['mean', 'min', 'max', 'std'])
print("\nAverage Pressure Error by Well:")
print(well_stats.round(2))


Average Pressure Error by Well:
          mean    min     max     std
well_id                              
W-101    14.36 -605.7  2129.4  272.34
W-102    32.56 -528.7  2922.2  349.66
W-103    -4.69 -546.4   583.6  197.81
W-104    11.94 -656.2  2612.0  320.13
W-105    -0.16 -506.9   517.9  202.02


## Key Findings

After removing non-operational zero-pressure intervals, W-102 showed the largest average pressure deviation at +32.56 psi. This suggests W-102 may require closer monitoring for over-pressure behavior. W-105 and W-103 were the most stable wells, staying closest to planned pressure targets.

The original analysis incorrectly suggested W-104 was the worst-performing well, but that was caused by zero-pressure downtime intervals. Removing those rows produced a more accurate performance comparison.

## Conclusion

W-102 shows the highest pressure variability and may require operational review.
W-105 is the most stable well and can serve as a benchmark.